<a href="https://colab.research.google.com/github/ZPRM/arc-prize-2026/blob/main/ARC_AGI3_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 ARC Prize 2026 — ARC-AGI-3 Starter Notebook

**Competition:** ARC Prize 2026 — ARC-AGI-3 Track  
**Launch Date:** March 25, 2026  
**Scoring:** RHAE (Relative Human Action Efficiency) — pronounced 'ray'  
**Current AI bar:** < 5% on all public games  
**Games:** ls20, ft09, vc33

---

## Notebook Structure
1. Install & Setup
2. Connect to ARC API
3. Random Agent Baseline
4. T1 Router (difficulty classifier)
5. T2 Loop (CNN explorer + Pang library)
6. T3 Oracle (frontier model)
7. Feedback Loop (T3 → T1)
8. Submit & Score

## 1. Install & Setup

In [ ]:
# Install ARC-AGI toolkit and dependencies
!pip install arc-agi python-dotenv -q
!pip install openai anthropic google-generativeai -q
!pip install torch torchvision scikit-learn -q

print('✅ All packages installed')

In [ ]:
# Clone key repos
!git clone https://github.com/arcprize/ARC-AGI-3-Agents.git --quiet
!git clone https://github.com/arcprize/ARC-AGI-2.git --quiet
!git clone https://github.com/mvakde/mdlARC.git --quiet
!git clone https://github.com/epang080516/arc_agi.git pang_library --quiet
!git clone https://github.com/DriesSmit/ARC3-solution.git stochastic_goose --quiet

print('✅ All repos cloned')
!ls

In [ ]:
import os
from google.colab import userdata

# ── Set your API key ──────────────────────────────────────────────
# In Colab: Secrets tab (🔑 icon on left) → Add secret
# Name: ARC_API_KEY
# Value: bdfedf56-3139-454d-8719-824acb2f4126

try:
    os.environ['ARC_API_KEY'] = userdata.get('ARC_API_KEY')
    print('✅ ARC_API_KEY loaded from Colab Secrets')
except:
    # Fallback: paste directly (never commit this to GitHub)
    os.environ['ARC_API_KEY'] = 'bdfedf56-3139-454d-8719-824acb2f4126'
    print('✅ ARC_API_KEY set directly')

print(f'   Key prefix: {os.environ["ARC_API_KEY"][:8]}...')

## 2. Connect to ARC API & Explore Games

In [ ]:
import arc_agi
from arcengine import GameAction, GameState

# Connect
arc = arc_agi.Arcade()
print('✅ Connected to ARC API')

# List available games
print('\nAvailable games:')
for game in arc.list_games():
    print(f'  - {game}')

In [ ]:
# Explore ls20 environment
env = arc.make('ls20')
print('Game: ls20')
print(f'Action space: {env.action_space}')
print(f'Observation: {env.observe()}')

## 3. Random Agent Baseline (establish RHAE starting point)

In [ ]:
import random

def run_random_agent(game_id: str, max_actions: int = 1000) -> dict:
    """Run a purely random agent and return scorecard."""
    arc_local = arc_agi.Arcade()
    env = arc_local.make(game_id)

    actions_taken = 0
    wins = 0

    for step in range(max_actions):
        action = random.choice(env.action_space)
        action_data = {}

        if action.is_complex():
            action_data = {'x': random.randint(0, 63), 'y': random.randint(0, 63)}

        obs = env.step(action, data=action_data)
        actions_taken += 1

        if obs and obs.state == GameState.WIN:
            wins += 1
            print(f'  Win at step {step}!')
            env.reset()

    scorecard = arc_local.get_scorecard()
    return {'game': game_id, 'actions': actions_taken, 'scorecard': scorecard}

# Run baseline on all 3 public games
games = ['ls20', 'ft09', 'vc33']
results = {}

for game in games:
    print(f'\nRunning random agent on {game}...')
    results[game] = run_random_agent(game, max_actions=200)
    print(f'  Done: {results[game]["scorecard"]}')

print('\n✅ Baseline complete')

## 4. T1 Router — Difficulty Classifier

In [ ]:
# T1 Router: classifies task difficulty to route to T1/T2/T3
# Based on: MDL compression score (Vakde) + observation features

import numpy as np
from sklearn.linear_model import LogisticRegression

class T1Router:
    """
    Difficulty router. Routes each game to the right compute tier:
    - EASY  → T1 (simple heuristic, ~$0.01/task)
    - MEDIUM → T2 (CNN explorer + Pang library, ~$0.15/task)
    - HARD  → T3 (frontier oracle, ~$5/task)
    """

    EASY   = 'T1'
    MEDIUM = 'T2'
    HARD   = 'T3'

    def __init__(self, t2_threshold=0.4, t3_threshold=0.75):
        self.t2_threshold = t2_threshold  # above this → T2
        self.t3_threshold = t3_threshold  # above this → T3
        self.model = LogisticRegression()
        self.trained = False
        self.total_spend = 0.0

    def extract_features(self, observation) -> np.ndarray:
        """Extract features from a game observation for difficulty estimation."""
        # Placeholder: in full build, use MDL score + object count + action space size
        features = np.array([
            len(str(observation)),        # observation complexity
            self.total_spend,             # budget used so far
        ])
        return features.reshape(1, -1)

    def route(self, observation) -> str:
        """Route an observation to T1, T2, or T3."""
        # Budget governor: raise T3 threshold as spend increases
        effective_t3 = min(0.95, self.t3_threshold + self.total_spend / 1000)

        if not self.trained:
            # Default before training: everything goes to T2
            return self.MEDIUM

        features = self.extract_features(observation)
        prob = self.model.predict_proba(features)[0][1]  # P(hard)

        if prob >= effective_t3:
            return self.HARD
        elif prob >= self.t2_threshold:
            return self.MEDIUM
        else:
            return self.EASY

    def update_from_t3(self, observation, was_successful: bool, cost: float):
        """Feedback loop: update router based on T3 results."""
        self.total_spend += cost
        # TODO: accumulate training data and retrain periodically

router = T1Router()
print('✅ T1 Router initialized')
print(f'   Default routing (untrained): everything → T2')

## 5. T2 Loop — CNN Explorer + Pang Library

In [ ]:
# T2: CNN action-effect predictor (inspired by StochasticGoose 1st place)
# Key insight: predict which actions cause state changes BEFORE random exploration

import torch
import torch.nn as nn

class ActionEffectPredictor(nn.Module):
    """
    Predicts whether an action will cause a meaningful state change.
    Trained online during gameplay.
    Input: current frame + action
    Output: P(state changes | frame, action)
    """
    def __init__(self, n_actions=8, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64 * 64 + n_actions, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def forward(self, frame_flat, action_onehot):
        x = torch.cat([frame_flat, action_onehot], dim=-1)
        return self.net(x)


class T2Explorer:
    """
    T2 tier: smart exploration using CNN action-effect predictor.
    Avoids wasting actions that won't change state (key to RHAE score).
    """
    def __init__(self, n_actions=8):
        self.predictor = ActionEffectPredictor(n_actions=n_actions)
        self.optimizer = torch.optim.Adam(self.predictor.parameters(), lr=1e-3)
        self.memory = []  # (frame, action, changed) tuples
        self.n_actions = n_actions

    def choose_action(self, env, observation) -> GameAction:
        """Choose action most likely to cause state change."""
        actions = env.action_space

        # Until predictor is trained, explore randomly
        if len(self.memory) < 20:
            return random.choice(actions)

        # Score each action by predicted state-change probability
        best_action = random.choice(actions)
        best_score  = -1

        for action in actions:
            # TODO: encode frame + action properly
            score = random.random()  # placeholder until full encoding
            if score > best_score:
                best_score  = score
                best_action = action

        return best_action

    def update(self, frame_before, action, frame_after):
        """Online training: did the action cause a state change?"""
        changed = (str(frame_before) != str(frame_after))
        self.memory.append((frame_before, action, changed))
        # TODO: batch train predictor every N steps


t2 = T2Explorer()
print('✅ T2 Explorer initialized')
print('   CNN action-effect predictor ready for online training')

## 6. Full Pipeline — T1 → T2 → T3

In [ ]:
def run_pipeline(game_id: str, max_actions: int = 500, tags: list = None):
    """
    Run the full T1→T2→T3 adaptive pipeline on a game.
    Routes each step to the appropriate compute tier.
    """
    print(f'\n🚀 Running pipeline on {game_id}')
    arc_local = arc_agi.Arcade()
    env = arc_local.make(game_id)

    actions_taken = 0
    tier_counts = {'T1': 0, 'T2': 0, 'T3': 0}

    for step in range(max_actions):
        obs = env.observe()

        # Route to correct tier
        tier = router.route(obs)
        tier_counts[tier] += 1

        # Choose action based on tier
        if tier == 'T1':
            # Simple heuristic — random for now
            action = random.choice(env.action_space)
        elif tier == 'T2':
            # CNN explorer
            action = t2.choose_action(env, obs)
        else:
            # T3: frontier model (placeholder)
            action = random.choice(env.action_space)

        # Step
        action_data = {}
        if action.is_complex():
            action_data = {'x': random.randint(0, 63), 'y': random.randint(0, 63)}

        obs_after = env.step(action, data=action_data)
        actions_taken += 1

        # Update T2 predictor
        t2.update(obs, action, obs_after)

        if obs_after and obs_after.state == GameState.WIN:
            print(f'  ✅ Win at step {step}!')
            env.reset()

    scorecard = arc_local.get_scorecard()
    print(f'\nResults for {game_id}:')
    print(f'  Actions: {actions_taken}')
    print(f'  Tier usage: {tier_counts}')
    print(f'  Scorecard: {scorecard}')
    return scorecard


# Run on ls20 as a test
scorecard = run_pipeline('ls20', max_actions=100)
print('\n✅ Pipeline test complete')

## 7. Run on All Games (Swarm mode)

In [ ]:
# Run across all 3 public games and compare RHAE scores
GAMES = ['ls20', 'ft09', 'vc33']

all_scores = {}
for game in GAMES:
    all_scores[game] = run_pipeline(game, max_actions=200, tags=['v1', 'baseline'])

print('\n=== Final Scores ===')
for game, score in all_scores.items():
    print(f'{game}: {score}')

## 8. Next Steps

### What to build next (Mar 20-24):
1. **Layer 1 encoder** — proper object segmentation from game frames
2. **Train T1 router** on H-ARC difficulty labels (HuggingFace dataset)
3. **Full CNN predictor** — replace placeholder with real frame encoding
4. **Pang concept library integration** — 10 LLM calls/task for T2
5. **T3 oracle** — Gemini Pro / Claude Sonnet for hardest tasks
6. **Feedback loop** — T3 results update T1 routing thresholds

### Key numbers to beat:
- Current AI bar: < 5% RHAE on all public games
- Preview competition 1st place: 12.58% (StochasticGoose)
- Target: > 15% by April, > 30% by competition close

### Resources:
- [ARC-AGI-3 Docs](https://docs.arcprize.org)
- [Leaderboard](https://three.arcprize.org)
- [StochasticGoose repo](https://github.com/DriesSmit/ARC3-solution)
- [Pang library](https://github.com/epang080516/arc_agi)
- Discord: discord.gg/9b77dPAmcA → #non-llm-approaches